# Step 3 gate — prior-ISMCTS vs heuristic (Pokémon TCG)

Runs the determinization-fix validation on the **real cg engine** (needs linux/amd64 — Colab is).
Both seats play the same deck; only the pilot differs (ISMCTS-with-archetype-prior vs the heuristic).
If search wins >50%, the fix is validated and R-NaD self-play (step 4) is worth the GPU.

**Before running:** upload the repo (or the minimal set) so this notebook can see:
`sdk/cg/` (incl. libcg.so), `autoresearch/agent_lucario.py`, `agent_ismcts_prior.py`,
`determinize.py`, `arch_priors.json`, `autoresearch/cards_full.csv`, a 60-card deck csv.
No GPU needed for this step (CPU engine).

### 1. Get the code onto the runtime
Easiest: zip the repo locally, upload here. Or `git clone` if it's pushed (private → use a token).

In [ ]:
# Option A: upload a zip of the repo, then unzip
from google.colab import files
up = files.upload()                       # pick pokemon.zip
import zipfile, os
z = next(iter(up))
zipfile.ZipFile(z).extractall('/content/repo')
# find the repo root (the dir containing sdk/ and autoresearch/)
root = None
for d,_,fs in os.walk('/content/repo'):
    if os.path.isdir(os.path.join(d,'sdk')) and os.path.isdir(os.path.join(d,'autoresearch')):
        root = d; break
print('repo root:', root)
os.chdir(root)

### 2. Verify the engine loads (linux/amd64 + libcg.so)

In [ ]:
import platform, sys
print('platform:', platform.platform(), '| machine:', platform.machine())
assert platform.machine() in ('x86_64','amd64'), 'cg engine is linux/amd64 — use a standard Colab CPU runtime'
sys.path.insert(0,'sdk')
from cg.game import battle_start, battle_select, battle_finish
print('libcg.so loaded OK')

### 3. Smoke test — one random self-play game completes

In [ ]:
import random
def read_deck(p): return [int(l) for l in open(p) if l.strip()][:60]
deck = read_deck('autoresearch/decks/lucario_meta.csv')
print('deck size', len(deck))
obs, sd = battle_start(deck, deck)
steps=0
while True:
    cur=obs['current']
    if cur is not None and cur.get('result',-1)!=-1:
        print('winner', cur['result'], 'in', steps, 'steps'); break
    sel=obs['select']
    if sel is None: obs=battle_select(read_deck('autoresearch/decks/lucario_meta.csv')); continue
    obs=battle_select(random.sample(range(len(sel['option'])), sel['maxCount'])); steps+=1
    if steps>20000: print('max steps'); break
battle_finish()

### 4. Run the gate
`--games 100` is a solid signal; start with 30 to time it, then scale. ISMCTS is CPU-heavy (sims×depth engine calls per decision), so expect minutes.

In [ ]:
!PYTHONPATH=sdk python run_gate.py --games 100 --sims 48 --depth 12

### 5. Download the result
Send `step3_result.json` back to the assistant — it reads that to pick up (verdict + per-game log).

In [ ]:
from google.colab import files
files.download('step3_result.json')
import json; print(json.dumps(json.load(open('step3_result.json')), indent=1)[:800])